## Runtime requirement

This notebook assumes a Google Colab runtime with GPU.

Before connecting, go to:
Runtime → Change runtime type → T4 GPU (for example)

In [1]:
# @title GPU availability check
import torch
from IPython.display import display, HTML

# Show a visible warning message if GPU is not available
if not torch.cuda.is_available():
    display(HTML("""
    <div style="padding:12px; border-radius:8px; background-color:#fff3cd; color:#856404; border:1px solid #ffeeba;">
        <b>Warning:</b> GPU is not enabled.<br>
        Please go to <b>Runtime -> Change runtime type -> T4 GPU</b>.
    </div>
    """))
else:
    print("GPU is available:", torch.cuda.get_device_name(0))

GPU is available: Tesla T4


## Installation requirements

In [2]:
!pip install biopython fair-esm -q
!pip install freesasa -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 84.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 93.1/93.1 kB 11.0 MB/s eta 0:00:00
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 270.1/270.1 kB 20.0 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done


## Importing modules

In [12]:
# Clone the GitHub repository into Colab
!git clone https://github.com/fukasawa-group/prsurflm
%cd /content/prsurflm

Cloning into 'prsurflm'...
remote: Enumerating objects: 28, done.
remote: Counting objects: 100% (28/28), done.
remote: Compressing objects: 100% (24/24), done.
remote: Total 28 (delta 3), reused 23 (delta 1), pack-reused 0 (from 0)
Receiving objects: 100% (28/28), 33.75 MiB | 18.83 MiB/s, done.
Resolving deltas: 100% (3/3), done.
/content/prsurflm


In [4]:
import os, sys
import argparse
sys.path.append("/content/prsurflm/")

import json
import numpy as np
import torch

print("imports OK")

imports OK


In [5]:
# @title ESM-2 preloading
# ESM-2 loading
import esm, torch

model_name = "esm2_t33_650M_UR50D"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

fn = getattr(esm.pretrained, model_name)
_model, _alphabet = fn()          # 1.5 GB
_model = _model.to(device).eval()
_batch_converter = _alphabet.get_batch_converter()

print(f"ESM-2 loaded on {device}  |  layers={_model.num_layers}")
del _model, _alphabet, _batch_converter

Downloading: "https://dl.fbaipublicfiles.com/fair-esm/models/esm2_t33_650M_UR50D.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D.pt
Downloading: "https://dl.fbaipublicfiles.com/fair-esm/regression/esm2_t33_650M_UR50D-contact-regression.pt" to /root/.cache/torch/hub/checkpoints/esm2_t33_650M_UR50D-contact-regression.pt
ESM-2 loaded on cuda  |  layers=33


In [25]:
# @title Data input
import os
import glob
import ipywidgets as widgets
from IPython.display import display, clear_output

# --------------------------------------------------
# Directories
# --------------------------------------------------
REPO_DIR = "/content/prsurflm"

# Bundled test data directories in the repository
TEST_ROOT_DIR = os.path.join(REPO_DIR, "test")
TEST_PDB_DIR = os.path.join(TEST_ROOT_DIR, "pdb")
TEST_DMASIF_DIR = os.path.join(TEST_ROOT_DIR, "dmasif")

# User-uploaded data directories in Colab
UPLOAD_PDB_DIR = "/content/pdb"
UPLOAD_DMASIF_DIR = "/content/dmasif"

os.makedirs(UPLOAD_PDB_DIR, exist_ok=True)
os.makedirs(UPLOAD_DMASIF_DIR, exist_ok=True)

# --------------------------------------------------
# Shared state for downstream inference
# --------------------------------------------------
input_state = {
    "mode": "demo",
    "base_name": None,
    "selected_pdb_path": None,
    "selected_chain_a_path": None,
    "selected_chain_b_path": None,
}

# --------------------------------------------------
# Helper functions
# --------------------------------------------------
def find_test_bases(test_pdb_dir, test_dmasif_dir):
    """Find valid test examples based on xxx.pdb and xxx_chain_A/B.npy"""
    pdb_files = glob.glob(os.path.join(test_pdb_dir, "*.pdb"))
    valid_bases = []

    for pdb_path in pdb_files:
        base = os.path.splitext(os.path.basename(pdb_path))[0]
        chain_a = os.path.join(test_dmasif_dir, f"{base}_chain_A.npy")
        chain_b = os.path.join(test_dmasif_dir, f"{base}_chain_B.npy")

        if os.path.exists(chain_a) and os.path.exists(chain_b):
            valid_bases.append(base)

    return sorted(valid_bases)


def set_demo_paths(base):
    """Set shared paths from bundled test data"""
    input_state["mode"] = "demo"
    input_state["base_name"] = base
    input_state["selected_pdb_path"] = os.path.join(TEST_PDB_DIR, f"{base}.pdb")
    input_state["selected_chain_a_path"] = os.path.join(TEST_DMASIF_DIR, f"{base}_chain_A.npy")
    input_state["selected_chain_b_path"] = os.path.join(TEST_DMASIF_DIR, f"{base}_chain_B.npy")


def clear_selected_paths():
    """Clear selected input paths"""
    input_state["base_name"] = None
    input_state["selected_pdb_path"] = None
    input_state["selected_chain_a_path"] = None
    input_state["selected_chain_b_path"] = None


def input_is_ready():
    """Check whether all required input paths are available"""
    return all([
        input_state["selected_pdb_path"],
        input_state["selected_chain_a_path"],
        input_state["selected_chain_b_path"],
    ])


# --------------------------------------------------
# Build demo mode UI
# --------------------------------------------------
test_bases = find_test_bases(TEST_PDB_DIR, TEST_DMASIF_DIR)

demo_title = widgets.HTML("<b>Bundled test data</b>")
demo_desc = widgets.HTML(
    "<p>Select one of the bundled examples included in the repository.</p>"
)

if test_bases:
    demo_selector = widgets.Dropdown(
        options=test_bases,
        value=test_bases[0],
        description="Example:",
        layout=widgets.Layout(width="450px")
    )
else:
    demo_selector = widgets.Dropdown(
        options=[],
        value=None,
        description="Example:",
        layout=widgets.Layout(width="450px")
    )

demo_status = widgets.Output()


def refresh_demo_status():
    """Refresh the demo mode status panel"""
    with demo_status:
        clear_output()

        if not test_bases:
            print("No valid bundled test examples were found.")
            print("Expected directories:")
            print(" -", TEST_PDB_DIR)
            print(" -", TEST_DMASIF_DIR)
            print("")
            print("Expected file pattern:")
            print(" - test/pdb/xxxx.pdb")
            print(" - test/dmasif/xxxx_chain_A.npy")
            print(" - test/dmasif/xxxx_chain_B.npy")
            clear_selected_paths()
            return

        base = demo_selector.value
        if base is None:
            print("No bundled test example is selected.")
            clear_selected_paths()
            return

        set_demo_paths(base)

        print("✅ Bundled test data selected")
        print("Base name:", base)
        print("PDB:", input_state["selected_pdb_path"])
        print("Chain A:", input_state["selected_chain_a_path"])
        print("Chain B:", input_state["selected_chain_b_path"])


def on_demo_change(change):
    """Handle changes in the demo selector"""
    if change["name"] == "value":
        refresh_demo_status()


demo_selector.observe(on_demo_change, names="value")

demo_ui = widgets.VBox([
    demo_title,
    demo_desc,
    demo_selector,
    demo_status,
])

# --------------------------------------------------
# Build upload mode UI
# --------------------------------------------------
upload_title = widgets.HTML("<b>Upload your own files</b>")
upload_desc = widgets.HTML(
    "<p>Required files:</p>"
    "<ul>"
    "<li><code>xxxx.pdb</code></li>"
    "<li><code>xxxx_chain_A.npy</code></li>"
    "<li><code>xxxx_chain_B.npy</code></li>"
    "</ul>"
    "<p>The main PDB file will be saved under <code>/content/pdb</code>, "
    "and the numpy files will be saved under <code>/content/dmasif</code>.</p>"
)

main_upload = widgets.FileUpload(
    accept=".pdb",
    multiple=False,
    description="Main PDB"
)

chain_upload = widgets.FileUpload(
    accept=".npy",
    multiple=True,
    description="Chain npy"
)

upload_status = widgets.Output()


def handle_main_upload(change):
    """Handle upload of the main PDB file"""
    with upload_status:
        clear_output()

        if not main_upload.value:
            print("No main PDB selected.")
            return

        item = list(main_upload.value.values())[0]
        filename = item["metadata"]["name"]
        content = item["content"]

        if not filename.lower().endswith(".pdb"):
            print("Error: Please upload a .pdb file.")
            return

        save_path = os.path.join(UPLOAD_PDB_DIR, filename)
        with open(save_path, "wb") as f:
            f.write(content)

        base = os.path.splitext(filename)[0]

        input_state["mode"] = "upload"
        input_state["base_name"] = base
        input_state["selected_pdb_path"] = save_path
        input_state["selected_chain_a_path"] = None
        input_state["selected_chain_b_path"] = None

        print("✅ Main PDB uploaded")
        print("File:", filename)
        print("Saved to:", save_path)
        print("")
        print("Now upload the matching files:")
        print(f" - {base}_chain_A.npy")
        print(f" - {base}_chain_B.npy")


def handle_chain_upload(change):
    """Handle upload of chain numpy files"""
    with upload_status:
        clear_output()

        base = input_state["base_name"]

        if base is None:
            print("Please upload the main PDB file first.")
            return

        if not chain_upload.value:
            print("No chain numpy files selected.")
            return

        required = {f"{base}_chain_A.npy", f"{base}_chain_B.npy"}
        uploaded = set()

        for item in chain_upload.value.values():
            filename = item["metadata"]["name"]
            content = item["content"]

            if not filename.lower().endswith(".npy"):
                print(f"Skipping non-npy file: {filename}")
                continue

            save_path = os.path.join(UPLOAD_DMASIF_DIR, filename)
            with open(save_path, "wb") as f:
                f.write(content)

            uploaded.add(filename)

        missing = sorted(required - uploaded)

        if not missing:
            input_state["selected_chain_a_path"] = os.path.join(UPLOAD_DMASIF_DIR, f"{base}_chain_A.npy")
            input_state["selected_chain_b_path"] = os.path.join(UPLOAD_DMASIF_DIR, f"{base}_chain_B.npy")

            print("✅ Chain numpy files uploaded successfully")
            print("PDB:", input_state["selected_pdb_path"])
            print("Chain A:", input_state["selected_chain_a_path"])
            print("Chain B:", input_state["selected_chain_b_path"])
        else:
            print("⚠ Missing required files.")
            print("Expected:")
            for f in sorted(required):
                print(" -", f)
            print("")
            print("Missing:")
            for f in missing:
                print(" -", f)


main_upload.observe(handle_main_upload, names="value")
chain_upload.observe(handle_chain_upload, names="value")

upload_ui = widgets.VBox([
    upload_title,
    upload_desc,
    widgets.HTML("<b>Step 1: Upload main PDB</b>"),
    main_upload,
    widgets.HTML("<b>Step 2: Upload chain numpy files (A and B)</b>"),
    chain_upload,
    upload_status,
])

# --------------------------------------------------
# Top-level mode selector
# --------------------------------------------------
title = widgets.HTML("<h3>Input Data</h3>")
desc = widgets.HTML(
    "<p>Choose one of the following options:</p>"
    "<ul>"
    "<li><b>Use bundled test data</b> (recommended for first-time users)</li>"
    "<li><b>Upload your own files</b></li>"
    "</ul>"
)

mode_selector = widgets.RadioButtons(
    options=[
        ("Use bundled test data", "demo"),
        ("Upload your own files", "upload"),
    ],
    value="demo",
    description="Mode:",
    layout=widgets.Layout(width="500px")
)

mode_status = widgets.Output()
mode_ui_box = widgets.VBox([])


def render_mode_ui(mode):
    """Render UI for the selected input mode"""
    if mode == "demo":
        mode_ui_box.children = [demo_ui]
        refresh_demo_status()
    else:
        mode_ui_box.children = [upload_ui]
        clear_selected_paths()
        input_state["mode"] = "upload"
        with upload_status:
            clear_output()
            print("Upload mode selected.")
            print("Please upload:")
            print(" - xxxx.pdb")
            print(" - xxxx_chain_A.npy")
            print(" - xxxx_chain_B.npy")


def on_mode_change(change):
    """Handle switching between demo mode and upload mode"""
    if change["name"] != "value":
        return

    with mode_status:
        clear_output()
        if change["new"] == "demo":
            print("Demo mode selected.")
        else:
            print("Upload mode selected.")

    render_mode_ui(change["new"])


mode_selector.observe(on_mode_change, names="value")

# --------------------------------------------------
# Summary panel
# --------------------------------------------------
summary_title = widgets.HTML("<b>Current selection</b>")
summary_button = widgets.Button(description="Show selected input")
summary_output = widgets.Output()


def show_summary(_):
    """Show the currently selected input paths"""
    with summary_output:
        clear_output()

        print("Mode:", input_state["mode"])
        print("Base name:", input_state["base_name"])
        print("PDB path:", input_state["selected_pdb_path"])
        print("Chain A path:", input_state["selected_chain_a_path"])
        print("Chain B path:", input_state["selected_chain_b_path"])
        print("Ready:", input_is_ready())


summary_button.on_click(show_summary)

# --------------------------------------------------
# Display the full UI
# --------------------------------------------------
ui = widgets.VBox([
    title,
    desc,
    mode_selector,
    mode_status,
    mode_ui_box,
    widgets.HTML("<hr>"),
    summary_title,
    summary_button,
    summary_output,
])

display(ui)

# Initialize the default view
render_mode_ui(mode_selector.value)

## Configuration

In [31]:
if not input_is_ready():
    raise ValueError("Input files are not fully set. Please select bundled test data or upload all required files.")

# Selected input from the widget
PDB_PATH     = input_state["selected_pdb_path"]
CHAIN_A_PATH = input_state["selected_chain_a_path"]
CHAIN_B_PATH = input_state["selected_chain_b_path"]

# Derived path components
PDB_DIR        = os.path.dirname(PDB_PATH)
PDB            = os.path.basename(PDB_PATH)
DMASIF_NPY_DIR = os.path.dirname(CHAIN_A_PATH)

# Fixed settings
RECEPTOR   = "A"
LIGAND     = "B"
CHECKPOINT = "/content/prsurflm/checkpoint/nr_pc_esm.pth"
DEVICE     = "cuda:0"

print("PDB_PATH:", PDB_PATH)
print("CHAIN_A_PATH:", CHAIN_A_PATH)
print("CHAIN_B_PATH:", CHAIN_B_PATH)
print("CHECKPOINT:", CHECKPOINT)
print("DEVICE:", DEVICE)

PDB_PATH: /content/prsurflm/test/pdb/1bs1_1.pdb
CHAIN_A_PATH: /content/prsurflm/test/dmasif/1bs1_1_chain_A.npy
CHAIN_B_PATH: /content/prsurflm/test/dmasif/1bs1_1_chain_B.npy
CHECKPOINT: /content/prsurflm/checkpoint/nr_pc_esm.pth
DEVICE: cuda:0


In [32]:
!python infer_pdb_pair.py \
    --pdb            {PDB_DIR}/{PDB} \
    --receptor       {RECEPTOR} \
    --ligand         {LIGAND} \
    --checkpoint     {CHECKPOINT} \
    --device         {DEVICE} \
    --dmasif_npy_dir {DMASIF_NPY_DIR} \
    --esm_pooling --esm_crossattn


[preprocess] /content/prsurflm/test/pdb/1bs1_1.pdb
  Receptor chains: ['A']  Ligand chains: ['B']
  Raw atoms: 2051 receptor, 2051 ligand
  Interface residues: 72 receptor, 72 ligand
  Interface atoms:    663 receptor, 663 ligand
  Selected atoms: 500 receptor, 500 ligand
  Loaded dMaSIF: /content/prsurflm/test/dmasif/1bs1_1_chain_A.npy  shape=(20921, 19)
  Loaded dMaSIF: /content/prsurflm/test/dmasif/1bs1_1_chain_B.npy  shape=(20933, 19)
  After dMaSIF append: (1000, 52)
  Output shape: (1000, 52)
Total residues: 224 receptor, 224 ligand
Interface residues (distance-based): 72 receptor, 72 ligand
Calculating SASA changes...
SASA filtering: 72 -> 37 receptor residues
SASA filtering: 72 -> 37 ligand residues
[ESM interface] distance+SASA: 37 receptor, 37 ligand residues
{
  "pdb": "/content/prsurflm/test/pdb/1bs1_1.pdb",
  "receptor": "A",
  "ligand": "B",
  "checkpoint": "/content/prsurflm/checkpoint/nr_pc_esm.pth",
  "device": "cuda:0",
  "in_channels": 49,
  "npoint": 1000,
  "used_